# **04 Model Training**

## Objectives

* Build a convolutional neural network (CNN) to classify cherry-leaf images.

* Train the model to distinguish healthy leaves from powdery mildew.

* Monitor training and validation performance.

* Save the trained model for evaluation and dashboard predictions.

* Work towards the client's target of at least 97% classification accuracy.

## Inputs

* Training images from data/processed/train/

* Validation images from data/processed/validation/

* Two image categories: healthy and powdery_mildew

## Outputs

* A trained binary classification model.

* Training and validation accuracy/loss histories.

* A saved model suitable for use in Notebook 05 and the Streamlit dashboard.

## Additional Comments

* The test dataset will not be used during model training.

* Images will be resized to 100 × 100 pixels to reduce computational requirements.

* Model performance will be assessed against the 97% accuracy target.



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

'/workspaces/FInal-Project-01/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

'/workspaces/FInal-Project-01'

# Section 1

Section 1 content

In [4]:
from pathlib import Path
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)

2026-10-12 10:36:51.574286: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-12 10:36:59.682217: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.16.1


In [5]:
train_path = Path("data/processed/train")
validation_path = Path("data/processed/validation")

print("Training folder exists:", train_path.exists())
print("Validation folder exists:", validation_path.exists())

Training folder exists: True
Validation folder exists: True


In [6]:
IMAGE_SIZE = (100, 100)
BATCH_SIZE = 32
SEED = 42

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_path,
    labels="inferred",
    label_mode="binary",
    class_names=["healthy", "powdery_mildew"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_path,
    labels="inferred",
    label_mode="binary",
    class_names=["healthy", "powdery_mildew"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Class names:", train_dataset.class_names)

Found 2936 files belonging to 2 classes.
Found 626 files belonging to 2 classes.
Class names: ['healthy', 'powdery_mildew']


## Building the CNN Model

A Convolutional Neural Network (CNN) is designed to recognise patterns in images.

The model will learn visual features such as leaf edges, textures,
colours, and the white powdery patches associated with powdery mildew.

The model uses:
- Convolutional layers to identify visual features.
- Max-pooling layers to reduce image dimensions.
- Dense layers to make predictions.
- A sigmoid output layer for binary classification.

In [8]:
model = models.Sequential([
    layers.Input(shape=(100, 100, 3)),

    layers.Rescaling(1.0 / 255),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 100, 100, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 98, 98, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 49, 49, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 47, 47, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 23, 23, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 21, 21, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,638,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,731,905 (6.61 MB)

 Trainable params: 1,731,905 (6.61 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

Model compiled successfully!


---

# Section 2

## Model Training

The CNN will be trained using the training dataset and evaluated
against the validation dataset after each epoch.

Early stopping will be used to reduce overfitting and restore
the model weights from the best-performing epoch.

In [10]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20,
    callbacks=[early_stopping]
)

Epoch 1/20


/workspaces/FInal-Project-01/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 185ms/step - accuracy: 0.9046 - loss: 0.2130 - val_accuracy: 0.9936 - val_loss: 0.0249
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 16s 174ms/step - accuracy: 0.9949 - loss: 0.0288 - val_accuracy: 1.0000 - val_loss: 0.0045
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 176ms/step - accuracy: 0.9959 - loss: 0.0231 - val_accuracy: 1.0000 - val_loss: 0.0017
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 20s 174ms/step - accuracy: 0.9932 - loss: 0.0192 - val_accuracy: 0.9505 - val_loss: 0.0867
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 16s 178ms/step - accuracy: 0.9973 - loss: 0.0159 - val_accuracy: 1.0000 - val_loss: 3.5414e-04
Epoch 6/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 17s 180ms/step - accuracy: 0.9898 - loss: 0.0291 - val_accuracy: 1.0000 - val_loss: 0.0010
Epoch 7/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 17s 180ms/step - accuracy: 0.9983 - loss: 0.0063 - val_accuracy: 0.9984 - val_loss: 0.0024
Epoch 8/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 16s 175ms/step - accuracy: 1.0000 - loss: 3.7781e-04 - val_accurac

---

In [11]:
training_accuracy = history.history["accuracy"]
validation_accuracy = history.history["val_accuracy"]

training_loss = history.history["loss"]
validation_loss = history.history["val_loss"]

epochs_range = range(1, len(training_accuracy) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy chart
axes[0].plot(epochs_range, training_accuracy, label="Training Accuracy")
axes[0].plot(epochs_range, validation_accuracy, label="Validation Accuracy")
axes[0].set_title("Model Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)

# Loss chart
axes[1].plot(epochs_range, training_loss, label="Training Loss")
axes[1].plot(epochs_range, validation_loss, label="Validation Loss")
axes[1].set_title("Model Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

<Figure size 1400x500 with 2 Axes>

## Saving the Trained Model

The trained CNN is saved in Keras format so it can be loaded
later for model evaluation and dashboard predictions.

Early stopping restored the weights from the epoch with the
lowest validation loss.

In [12]:
from pathlib import Path

model_directory = Path("outputs/models")
model_directory.mkdir(parents=True, exist_ok=True)

model_path = model_directory / "cherry_leaf_cnn.keras"

model.save(model_path)

print("Model saved successfully!")
print("Location:", model_path)
print("File size:", round(model_path.stat().st_size / (1024 * 1024), 2), "MB")

Model saved successfully!
Location: outputs/models/cherry_leaf_cnn.keras
File size: 19.87 MB


In [13]:
import pandas as pd

history_df = pd.DataFrame(history.history)

history_path = model_directory / "training_history.csv"

history_df.to_csv(history_path, index=False)

print("Training history saved successfully!")
print("Location:", history_path)

display(history_df)

Training history saved successfully!
Location: outputs/models/training_history.csv


,accuracy,loss,val_accuracy,val_loss
0,0.904632,0.213029,0.993610,0.024944
1,0.994891,0.028802,1.000000,0.004462
2,0.995913,0.023145,1.000000,0.001720
3,0.993188,0.019192,0.950479,0.086690
4,0.997275,0.015950,1.000000,0.000354
5,0.989782,0.029051,1.000000,0.001032
6,0.998297,0.006346,0.998403,0.002397
7,1.000000,0.000378,1.000000,0.000832


In [14]:
# Load the saved model
loaded_model = tf.keras.models.load_model(
    "outputs/models/cherry_leaf_cnn.keras"
)

# Display its architecture
loaded_model.summary()

print("Saved model loaded successfully!")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 100, 100, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 98, 98, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 49, 49, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 47, 47, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 23, 23, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 21, 21, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,638,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,195,717 (19.82 MB)

 Trainable params: 1,731,905 (6.61 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 3,463,812 (13.21 MB)

Saved model loaded successfully!


## Conclusions and Next Steps

### Conclusions

- A Convolutional Neural Network was successfully built using TensorFlow and Keras.
- The model was trained using the training dataset and monitored using a separate validation dataset.
- Early stopping and dropout were used to reduce overfitting.
- Training and validation accuracy and loss were plotted to assess model learning.
- The trained model was saved as `outputs/models/cherry_leaf_cnn.keras`.
- The training history was saved as `outputs/models/training_history.csv`.
- The saved model was successfully reloaded for verification.

### Next Steps

- Load the saved CNN in Notebook 05.
- Evaluate its performance using the held-out test dataset.
- Generate a confusion matrix and classification report.
- Compare the test accuracy against the client's 97% requirement.
- Use the trained model in the Streamlit dashboard.

---